# Ders 3: Verimli Dikkat ve Uzun Bağlam

**İleri Derin Öğrenme** — Haydar Kılıç

Ön koşul: *Derin Öğrenme*, Ders 7 (Öz-dikkat, çok başlı dikkat).

Standart dikkat hem $O(n^2)$ zaman **hem de** $O(n^2)$ bellek harcar. Bu defterde uzun bağlamları
pratik hâle getiren teknikleri işliyoruz: FlashAttention'ın arkasındaki çevrimiçi softmax hilesi,
KV önbelleği sıkıştırma (MQA/GQA), dışarı doğru genelleyebilen göreli konum kodlamaları (RoPE,
ALiBi), seyrek dikkat desenleri ve doğrusal dikkat. Her biri sıfırdan kodlanıp tam dikkat ile
karşılaştırılarak doğrulanıyor.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams["figure.dpi"] = 100

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    s = Q @ K.T / np.sqrt(Q.shape[-1])
    if mask is not None:
        s = np.where(mask, s, -np.inf)
    return softmax(s) @ V, softmax(s)

print("Kütüphaneler yüklendi.")


## 1. Tam Dikkatin Maliyeti

Dizi uzunluğu $n$, başlık boyutu $d$ için skor matrisi $QK^\top$ $n \times n$ boyutundadır. Zaman
$O(n^2 d)$; pratikte asıl bağlayıcı kısıt olan aktivasyon belleği ise **katman ve başlık başına**
$O(n^2)$'dir, çünkü dikkat matrisi geri geçiş için saklanmak zorundadır.

$n = 32{.}768$ ve 32 başlıkta tek bir katmanın dikkat matrisleri fp16'da
$32 \times 32768^2 \times 2$ bayt $\approx 69$ GB eder. Naif dikkatin uzun bağlamda sadece
yavaşlamadığını — hiç sığmadığını — gösteren şey budur.


In [ ]:
n = np.logspace(2, 5, 200)
d, H, L, bytes_per = 128, 32, 32, 2

mem_scores = H * n**2 * bytes_per / 1e9              # tek katman, tüm başlıklar
mem_kv     = 2 * n * d * H * L * bytes_per / 1e9     # KV önbelleği, tüm katmanlar

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].loglog(n, mem_scores, lw=2, label="dikkat matrisi (1 katman, 32 başlık)")
axes[0].loglog(n, mem_kv,     lw=2, label="KV önbelleği (32 katman)")
axes[0].axhline(80, ls="--", c="crimson", lw=1.5, label="80 GB hızlandırıcı")
axes[0].set_xlabel("dizi uzunluğu n"); axes[0].set_ylabel("bellek (GB)")
axes[0].set_title("fp16 bellek kullanımı"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3, which="both")

axes[1].loglog(n, 4*n**2*d/1e9, lw=2, label="dikkat FLOP'ları  O(n^2 d)")
axes[1].loglog(n, 8*n*d**2/1e9, lw=2, label="izdüşüm FLOP'ları  O(n d^2)")
axes[1].set_xlabel("dizi uzunluğu n"); axes[1].set_ylabel("başlık başına GFLOP")
axes[1].set_title("Hesap"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3, which="both")

plt.tight_layout(); plt.show()
for nn in [1024, 8192, 32768, 131072]:
    print(f"n={nn:7d}: dikkat matrisleri {H*nn**2*2/1e9:8.1f} GB   KV önbelleği {2*nn*d*H*L*2/1e9:6.1f} GB")


## 2. Çevrimiçi Softmax — FlashAttention'ın Ardındaki Fikir

FlashAttention dikkatin matematiğini değiştirmez; **ara sonuçların nerede durduğunu** değiştirir.
Hızlandırıcılarda küçük ama son derece hızlı bir yonga içi SRAM ve büyük ama yavaş bir HBM vardır.
Naif dikkat $n \times n$ matrisin tamamını HBM'e yazıp geri okur — yani *bellek bant genişliğine
bağlıdır*, hesap gücüne değil.

İşi mümkün kılan hile, softmax'ın **artımlı** hesaplanabilmesidir. Anahtarları bloklar hâlinde işleyip
koşan bir maksimum $m$, koşan bir payda $\ell$ ve koşan bir çıktı $o$ tutulur. Maksimumu
$m_{\text{yeni}}$ olan yeni bir blok geldiğinde eldekiler yeniden ölçeklenir:

$$m' = \max(m, m_{\text{yeni}}), \qquad
\ell' = e^{m - m'}\ell + e^{m_{\text{yeni}} - m'}\ell_{\text{yeni}}, \qquad
o' = \frac{e^{m-m'}\ell\, o + e^{m_{\text{yeni}}-m'} \ell_{\text{yeni}} o_{\text{yeni}}}{\ell'}.$$

$n \times n$ matris hiçbir zaman oluşturulmaz: bellek $O(n)$ olur ve hızlanmayı üreten şey HBM
trafiğindeki azalmadır. Şimdi cebirin tam olarak doğru olduğunu doğrulayalım.


In [ ]:
def flash_attention(Q, K, V, block=16):
    n, d = Q.shape
    O = np.zeros((n, V.shape[1]))
    for i in range(n):                                   # sorgu döngüsü (gerçek çekirdek bunu da bloklar)
        m, l, o = -np.inf, 0.0, np.zeros(V.shape[1])
        for j0 in range(0, n, block):                    # anahtar blokları üzerinde akış
            Kb, Vb = K[j0:j0+block], V[j0:j0+block]
            s  = Kb @ Q[i] / np.sqrt(d)
            m_b = s.max()
            m_new = max(m, m_b)
            p = np.exp(s - m_new)
            l_new = np.exp(m - m_new)*l + p.sum()
            o = (np.exp(m - m_new)*l*o + p @ Vb) / l_new
            m, l = m_new, l_new
        O[i] = o
    return O

n, d = 64, 16
Q, K, V = (np.random.randn(n, d) for _ in range(3))

O_exact, _ = attention(Q, K, V)
O_flash    = flash_attention(Q, K, V, block=16)
print(f"maks |tam - flash| = {np.abs(O_exact - O_flash).max():.2e}   (kayan nokta hatası dışında aynı)")

# Ara sonuçların bellek ayak izi
ns = np.array([512, 2048, 8192, 32768])
fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(ns, ns**2*2/1e6, "o-", lw=2, label="standart: HBM'de O(n^2) skor")
ax.loglog(ns, ns*d*2*3/1e6, "s-", lw=2, label="flash: O(n) koşan istatistik")
ax.set_xlabel("dizi uzunluğu n"); ax.set_ylabel("ara sonuç belleği (MB)")
ax.set_title("Aynı çıktı, farklı bellek karmaşıklığı")
ax.legend(fontsize=9); ax.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()


## 3. KV Önbelleği: MHA → MQA → GQA

Otoregresif üretim sırasında önceki tüm token'ların anahtar ve değerleri önbelleğe alınır; böylece her
yeni token $O(n^2)$ yerine $O(n)$ maliyetli olur. Önbellek boyutu

$$2 \cdot n \cdot L \cdot H \cdot d_{\text{başlık}} \cdot \text{bayt}$$

kadardır ve uzun bağlamda model ağırlıklarını fazlasıyla aşar. Üretim her tek token için önbelleğin
tamamını okuduğundan, kod çözme bellek bant genişliğine bağlıdır — yani önbellek boyutu doğrudan
gecikmenin *kendisidir*.

- **MHA:** $H$ sorgu başlığı, $H$ anahtar/değer başlığı.
- **MQA:** $H$ sorgu başlığı, **1** paylaşılan anahtar/değer başlığı. $H$ kat küçük önbellek, bir
  miktar kalite kaybı.
- **GQA:** $H$ sorgu başlığı, $G$ anahtar/değer grubu ($1 < G < H$). Güncel açık modellerin çoğunun
  tercih ettiği orta yol.


In [ ]:
# Q_heads: (H, n, d).  K_groups / V_groups: (G, n, d).
# h numaralı sorgu başlığı  h // (H//G)  numaralı anahtar/değer grubunu okur.
def grouped_attention(Q_heads, K_groups, V_groups):
    H, n, d = Q_heads.shape
    G = K_groups.shape[0]
    per = H // G
    return np.stack([attention(Q_heads[h], K_groups[h//per], V_groups[h//per])[0] for h in range(H)])

H, n, d = 8, 32, 16
Qh = np.random.randn(H, n, d)
Kf, Vf = np.random.randn(H, n, d), np.random.randn(H, n, d)

configs = {"MHA (G=8)": 8, "GQA (G=4)": 4, "GQA (G=2)": 2, "MQA (G=1)": 1}
ref = grouped_attention(Qh, Kf, Vf)
print(f"{'yapılandırma':14s} {'KV başlık':>10} {'önbellek (göreli)':>18} {'MHA\'ya göre sapma':>19}")
for name, G in configs.items():
    Kg = Kf.reshape(G, H//G, n, d).mean(1)          # KV başlıklarını ortalayarak grupla
    Vg = Vf.reshape(G, H//G, n, d).mean(1)
    out = grouped_attention(Qh, Kg, Vg)
    print(f"{name:12s} {G:9d} {G/H:17.3f} {np.abs(out-ref).mean():21.4f}")

L_layers, d_head, ctx = 32, 128, np.logspace(2, 5.2, 100)
plt.figure(figsize=(8.5, 4.2))
for name, G in configs.items():
    plt.loglog(ctx, 2*ctx*L_layers*G*d_head*2/1e9, lw=2, label=f"{name}")
plt.axhline(80, ls="--", c="crimson", lw=1.4, label="80 GB hızlandırıcı")
plt.xlabel("bağlam uzunluğu"); plt.ylabel("KV önbelleği (GB, fp16)")
plt.title("8 sorgu başlıklı, 32 katmanlı bir model için KV önbelleği")
plt.legend(fontsize=9); plt.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()


## 4. Konum: Mutlak, RoPE ve ALiBi

Öğrenilen mutlak gömmeler dışarı genelleyemez: eğitim 4096'da bittiyse 5000. konumun gömmesi yoktur.
Bunu iki kodlama çözdü.

**RoPE**, sorgu ve anahtarın her iki boyutlu dilimini konumla orantılı bir açıyla döndürür:

$$\tilde q_m = R_{\Theta, m} q_m, \qquad
\langle \tilde q_m, \tilde k_n \rangle = \langle R_{\Theta,\, m-n} q_m,\, k_n \rangle .$$

Böylece iç çarpım yalnızca **göreli** fark $m - n$'ye bağlı olur — tam da istediğimiz özellik — ve
bu, tek tek vektörler düzeyinde uygulanır.

**ALiBi** ise gömmeyi tamamen atlar ve skorlara doğrusal bir ceza ekler:
$s_{ij} \leftarrow s_{ij} - \lambda_h |i - j|$, her başlık için farklı bir eğim $\lambda_h$ ile.
Maliyeti sıfırdır ve zarifçe dışarı genelleşir; bedeli ise sabit kodlanmış bir "yakınlık" önselidir.


In [ ]:
def rope(x, base=10000):
    n, d = x.shape
    pos   = np.arange(n)[:, None]
    theta = base ** (-np.arange(0, d, 2)/d)[None, :]
    ang   = pos * theta
    cos, sin = np.cos(ang), np.sin(ang)
    x1, x2 = x[:, 0::2], x[:, 1::2]
    out = np.empty_like(x)
    out[:, 0::2] = x1*cos - x2*sin
    out[:, 1::2] = x1*sin + x2*cos
    return out

# Özellik kontrolü: RoPE iç çarpımı yalnızca (m - n)'ye bağlıdır
n, d = 64, 32
q = np.tile(np.random.randn(1, d), (n, 1))       # her konumda aynı içerik
k = np.tile(np.random.randn(1, d), (n, 1))
qr, kr = rope(q), rope(k)
S = qr @ kr.T
offsets = [(S[i, j], i-j) for i in range(n) for j in range(n)]
by_off  = {}
for v, o in offsets:
    by_off.setdefault(o, []).append(v)
spread = max(np.ptp(v) for v in by_off.values())
print(f"sabit bir fark (m-n) içindeki skor yayılımının maksimumu: {spread:.2e}  -> yalnızca m-n'ye bağlı")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
im = axes[0].imshow(S, cmap="RdBu_r"); plt.colorbar(im, ax=axes[0])
axes[0].set_title("RoPE skor matrisi: köşegenler boyunca sabit")
axes[0].set_xlabel("anahtar konumu n"); axes[0].set_ylabel("sorgu konumu m")

offs = np.array(sorted(by_off)); vals = np.array([np.mean(by_off[o]) for o in offs])
axes[1].plot(offs, vals, lw=2)
axes[1].set_xlabel("göreli fark m - n"); axes[1].set_ylabel("skor")
axes[1].set_title("Skor yalnızca göreli farkın fonksiyonu"); axes[1].grid(alpha=0.3)

heads = 8
slopes = 2.0 ** (-8.0*np.arange(1, heads+1)/heads)
i = np.arange(64)
for h in range(heads):
    axes[2].plot(i, -slopes[h]*i, lw=1.6, label=f"başlık {h+1}" if h in (0, heads-1) else None)
axes[2].set_xlabel("uzaklık |i - j|"); axes[2].set_ylabel("skora eklenen ALiBi sapması")
axes[2].set_title("ALiBi: başlık başına bir doğrusal yakınlık eğimi")
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("ALiBi eğimleri:", slopes.round(4))


## 5. Seyrek ve Kayan Pencereli Dikkat

Her token her şey yerine $w$ komşuluk penceresine bakarsa maliyet $O(nw)$'ye düşer. Bariz itiraz —
model artık uzağı göremez — derinlikle yanıtlanır: $L$ katmandan sonra alıcı alan her yönde
$\approx L \cdot w/2$ olur, tıpkı bir ESA'daki gibi. Birkaç **global** token (ya da adımlı bir desen)
eklemek uzun menzilli yolları tek sıçramada geri kazandırır.


In [ ]:
n = 64
idx = np.arange(n)
patterns = {
    "tam":            np.ones((n, n), bool),
    "nedensel":          idx[:, None] >= idx[None, :],
    "kayan pencere (w=8)":   (np.abs(idx[:, None] - idx[None, :]) <= 4) & (idx[:, None] >= idx[None, :]),
    "genişletilmiş (2)":     ((idx[:, None] - idx[None, :]) % 2 == 0) & (idx[:, None] >= idx[None, :]),
    "pencere + global": (((np.abs(idx[:, None]-idx[None, :]) <= 4) | (idx[None, :] < 4) | (idx[:, None] < 4))
                        & (idx[:, None] >= idx[None, :])),
}

fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
for ax, (name, m) in zip(axes, patterns.items()):
    ax.imshow(m, cmap="Blues", interpolation="nearest")
    ax.set_title(f"{name}\ngirdilerin %{m.mean()*100:.0f}'i", fontsize=10)
    ax.set_xlabel("anahtar"); ax.set_ylabel("sorgu")
plt.tight_layout(); plt.show()

# Kayan pencerede alıcı alanın derinlikle büyümesi
w, L = 8, np.arange(1, 25)
plt.figure(figsize=(8, 4))
plt.plot(L, L*w//2, "o-", lw=2, label=f"kayan pencere w={w}")
plt.axhline(4096, ls="--", c="crimson", label="hedef bağlam 4096")
plt.plot(L, L*w//2*0 + 4096, alpha=0)
plt.yscale("log"); plt.xlabel("katman"); plt.ylabel("alıcı alan (token)")
plt.title("Derinlik yerel bir pencereyi global bir alıcı alana dönüştürür")
plt.legend(fontsize=9); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"w={w} ile 4096 token'lık alıcı alana ulaşmak ~{int(np.ceil(4096/(w/2)))} katman gerektirir "
      f"-- kayan pencereli modellerin global token eklemesinin nedeni budur.")


## 6. Doğrusal Dikkat: Softmax'ı Birleşme Özelliğiyle Takas Etmek

$n \times n$ matrisi var olmaya zorlayan şey softmax'tır. $\exp(q^\top k)$ yerine çarpanlarına
ayrılabilen bir çekirdek $\phi(q)^\top \phi(k)$ koyarsak matris çarpımı birleşmeli hâle gelir ve
sırayı değiştirebiliriz:

$$\underbrace{(\phi(Q)\phi(K)^\top)V}_{O(n^2 d)} \;=\; \underbrace{\phi(Q)(\phi(K)^\top V)}_{O(n d^2)} .$$

İkinci biçim hiçbir zaman $n \times n$ bir nesne kurmaz. Nedensel modellerde içteki terim koşan bir
toplama — yani $d \times d$ boyutunda bir **yinelemeli duruma** — dönüşür; bu da dikkat ile modern
durum-uzayı / doğrusal-RNN modelleri arasındaki köprüdür: adım başına sabit bellek, toplamda doğrusal
maliyet; bedeli ise tüm geçmişi sıkıştırmak zorunda olan sonlu kapasiteli bir durumdur.


In [ ]:
phi = lambda x: np.maximum(x, 0) + 1e-6          # elu+1 tarzı pozitif öznitelik eşlemesi

def linear_attention_quadratic(Q, K, V):         # (phi(Q) phi(K)^T) V
    A = phi(Q) @ phi(K).T
    return (A / A.sum(1, keepdims=True)) @ V

def linear_attention_factored(Q, K, V):          # phi(Q) (phi(K)^T V)
    Kp = phi(K)
    S  = Kp.T @ V                                # d x d_v durum
    z  = Kp.sum(0)                               # d normalleştirici
    num = phi(Q) @ S
    den = phi(Q) @ z
    return num / den[:, None]

def linear_attention_recurrent(Q, K, V):         # nedensel, adım başına O(1) bellek
    d, dv = K.shape[1], V.shape[1]
    S, z = np.zeros((d, dv)), np.zeros(d)
    out = np.zeros((Q.shape[0], dv))
    for t in range(Q.shape[0]):
        kt = phi(K[t])
        S += np.outer(kt, V[t]); z += kt
        out[t] = (phi(Q[t]) @ S) / (phi(Q[t]) @ z + 1e-9)
    return out

n, d = 48, 16
Q, K, V = np.random.randn(n, d), np.random.randn(n, d), np.random.randn(n, d)
o1, o2 = linear_attention_quadratic(Q, K, V), linear_attention_factored(Q, K, V)
print(f"kuadratik ve çarpanlı biçim: maks fark = {np.abs(o1-o2).max():.2e}  (matematiksel olarak özdeş)")

causal = np.arange(n)[:, None] >= np.arange(n)[None, :]
Ap = phi(Q) @ phi(K).T * causal
o_causal = (Ap / Ap.sum(1, keepdims=True)) @ V
print(f"nedensel paralel ve yinelemeli biçim: maks fark = "
      f"{np.abs(o_causal - linear_attention_recurrent(Q, K, V)).max():.2e}")

ns = np.logspace(2, 5, 60); dh = 64
plt.figure(figsize=(8.5, 4.2))
plt.loglog(ns, 4*ns**2*dh, lw=2, label="softmax dikkat  O(n^2 d)")
plt.loglog(ns, 4*ns*dh**2, lw=2, label="doğrusal dikkat   O(n d^2)")
plt.axvline(dh, ls=":", c="crimson"); plt.text(dh*1.15, 1e8, "n = d", c="crimson", fontsize=9)
plt.xlabel("dizi uzunluğu n"); plt.ylabel("FLOPs"); plt.legend(fontsize=9)
plt.title(f"Kesişim n ~ d civarında (burada d={dh}): doğrusal dikkat ancak uzun dizilerde kazandırır")
plt.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()

# Kaybedilen: softmax tek bir token'a yığılabilir, pozitif çekirdek bunu zor yapar
s = np.linspace(-6, 6, 200)
plt.figure(figsize=(8, 3.6))
plt.plot(s, softmax(np.stack([s, np.zeros_like(s)]).T)[:, 0], lw=2, label="A token'ına softmax ağırlığı")
lin = phi(s)/(phi(s) + phi(np.zeros_like(s)))
plt.plot(s, lin, lw=2, label="A token'ına doğrusal çekirdek ağırlığı")
plt.xlabel("A ve B token'ları arasındaki skor farkı"); plt.ylabel("dikkat ağırlığı")
plt.title("Softmax keskin olabilir; pozitif öznitelik eşlemesi çok daha yumuşaktır")
plt.legend(fontsize=9); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


Son şekil doğrusal dikkatin dürüst bedelidir: softmax kütlesinin neredeyse tamamını tek bir token'a
yığabilir (geri getirme benzeri davranış için gereklidir), oysa düşük boyutlu pozitif bir öznitelik
eşlemesi çok daha yayvan bir dağılım üretir. Doğrusal dikkat ve durum-uzayı modellerinin genellikle
birkaç tam dikkat katmanıyla **melezlenmesinin** nedeni budur; tam ikame edilmemelerinin de.

## 7. Özet

| Kavram | Açıklama |
|---|---|
| **Dikkatin maliyeti** | $O(n^2 d)$ zaman; katman ve başlık başına $O(n^2)$ aktivasyon belleği |
| **Çevrimiçi softmax** | Koşan $(m, \ell, o)$ güncellemesi; dikkati $O(n)$ bellekte *tam* hesaplar |
| **FlashAttention** | Bloklama + çevrimiçi softmax; kazancı FLOP'tan değil HBM trafiğinden gelir |
| **KV önbelleği** | $2nLHd$ değer; uzun bağlamda kod çözme gecikmesine hâkim olur |
| **MQA / GQA** | Anahtar/değer başlıklarını paylaştırır; önbellek $H/G$ kat küçülür |
| **RoPE** | q, k'yı konuma göre döndürür; iç çarpım yalnızca $m-n$'ye bağlıdır |
| **ALiBi** | Başlık başına doğrusal uzaklık cezası; bedava, genelleşir, yakınlık önseli dayatır |
| **Kayan pencere** | $O(nw)$; alıcı alan derinlikle büyür; global token'lar uzun sıçramaları geri getirir |
| **Doğrusal dikkat** | $\phi(Q)(\phi(K)^\top V)$: $O(nd^2)$, yinelemeli durum; ama çok daha yumuşak bir dikkat dağılımı |

**Sonraki Defter →** Öz-Denetimli ve Temsil Öğrenme
